
# Lab Sheet 5 – Retrieval-Augmented Generation (RAG) Architecture

## Objective
To develop a document-based question-answering chatbot using Retrieval-Augmented Generation (RAG).

## Technologies Used
- Python and Google Colab
- LangChain
- ChromaDB
- Hugging Face Embeddings
- Qwen2.5-0.5B-Instruct
- Gradio

## System Overview
The system loads a PDF document, extracts its text, splits it into chunks, and converts those chunks into embeddings stored in ChromaDB.

When a user asks a question, the system retrieves relevant document chunks and uses the Qwen language model to generate an answer.

A relevance filter helps reject questions unrelated to the uploaded document.


In [1]:

import torch

print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))


GPU Available: True
GPU Name: Tesla T4


In [2]:

%pip install -q langchain langchain-chroma chromadb langchain-text-splitters pypdf


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
t

In [3]:

%pip install -q "python-dateutil==2.9.0.post0" "opentelemetry-api>=1.42,<1.43" "oauthlib>=3.2.0,<4.0"


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-semantic-conventions 0.66b2 requires opentelemetry-api==1.45.2, but you have opentelemetry-api 1.42.1 which is incompatible.
opentelemetry-sdk 1.45.2 requires opentelemetry-api==1.45.2, but you have opentelemetry-api 1.42.1 which is incompatible.
kubernetes 37.0.2 requires oauthlib>=4.0.0, but you have oauthlib 3.3.1 which is incompatible.
kubernetes 37.0.2 requires python-dateutil>=3.0.0, but you have python-dateutil 2.9.0.post0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [4]:

import torch
import langchain
import chromadb
import pypdf

from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("All required libraries imported successfully!")
print("GPU Available:", torch.cuda.is_available())


All required libraries imported successfully!
GPU Available: True


In [5]:

%pip install -q langchain-community


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kubernetes 37.0.2 requires oauthlib>=4.0.0, but you have oauthlib 3.3.1 which is incompatible.
kubernetes 37.0.2 requires python-dateutil>=3.0.0, but you have python-dateutil 2.9.0.post0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [6]:

from langchain_community.document_loaders import PyPDFLoader

print("PDF Loader imported successfully!")


PDF Loader imported successfully!


/tmp/ipykernel_379/3906659437.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [7]:

from google.colab import files

uploaded = files.upload()

print("Uploaded files:", list(uploaded.keys()))


Saving Lab sheet 5.pdf to Lab sheet 5.pdf
Uploaded files: ['Lab sheet 5.pdf']


In [8]:

from langchain_community.document_loaders import PyPDFLoader

pdf_path = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("Total pages loaded:", len(documents))
print("\nSample extracted text:\n")
print(documents[0].page_content[:1000])


Total pages loaded: 6

Sample extracted text:

PUSL 3205-Large Language Models and Generative AI 
 
RAG architecture 
Setting up your python environment 
1. Create a folder in your machine 
2. Navigate to that folder using your terminal 
3. Install python 
4. Create virtual environment 
py -3.12 -m venv rag_env 
5. Activate virtual environment 
.\rag_env\Scripts\Activate.ps1 
6. Install required libraries 
  python -m pip install --upgrade pip 
pip install langchain langchain-community langchain-ollama langchain-
chroma chromadb streamlit pymupdf python-docx sentence-transformers 
python-dotenv 
 About Langchain 
LangChain is an open-source framework for developing applications powered by Large 
Language Models (LLMs). It helps developers connect LLMs with external data 
sources, embedding models, vector databases, retrievers, APIs and other tools. 
 
7. Check the chromaDB 
python -c "import chromadb; print ('ChromaDB installed successfully')" 
 
8. Save your dependencies 
pip freeze >

In [9]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total chunks created:", len(chunks))
print("\nSample chunk:\n")
print(chunks[0].page_content)


Total chunks created: 10

Sample chunk:

PUSL 3205-Large Language Models and Generative AI 
 
RAG architecture 
Setting up your python environment 
1. Create a folder in your machine 
2. Navigate to that folder using your terminal 
3. Install python 
4. Create virtual environment 
py -3.12 -m venv rag_env 
5. Activate virtual environment 
.\rag_env\Scripts\Activate.ps1 
6. Install required libraries 
  python -m pip install --upgrade pip 
pip install langchain langchain-community langchain-ollama langchain-
chroma chromadb streamlit pymupdf python-docx sentence-transformers 
python-dotenv 
 About Langchain 
LangChain is an open-source framework for developing applications powered by Large


In [10]:

%pip install -q langchain-huggingface sentence-transformers


In [11]:

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Load the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Store document chunks in ChromaDB
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="lab_sheet_5"
)

print("Embeddings created successfully!")
print("Documents stored in ChromaDB:", vectorstore._collection.count())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings created successfully!
Documents stored in ChromaDB: 10


In [12]:

# Create a retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

# Ask a question
question = "What is LangChain?"

# Retrieve relevant document chunks
retrieved_docs = retriever.invoke(question)

print("Question:", question)
print("\nRelevant chunks retrieved:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Result {i+1} ---")
    print("Page:", doc.metadata.get("page", 0) + 1)
    print(doc.page_content[:500])


Question: What is LangChain?

Relevant chunks retrieved: 3

--- Result 1 ---
Page: 1
LangChain is an open-source framework for developing applications powered by Large 
Language Models (LLMs). It helps developers connect LLMs with external data 
sources, embedding models, vector databases, retrievers, APIs and other tools. 
 
7. Check the chromaDB 
python -c "import chromadb; print ('ChromaDB installed successfully')" 
 
8. Save your dependencies 
pip freeze > requirements.txt

--- Result 2 ---
Page: 1
PUSL 3205-Large Language Models and Generative AI 
 
RAG architecture 
Setting up your python environment 
1. Create a folder in your machine 
2. Navigate to that folder using your terminal 
3. Install python 
4. Create virtual environment 
py -3.12 -m venv rag_env 
5. Activate virtual environment 
.\rag_env\Scripts\Activate.ps1 
6. Install required libraries 
  python -m pip install --upgrade pip 
pip install langchain langchain-community langchain-ollama langchain-
chroma chromadb stre

In [13]:

%pip install -q transformers accelerate


In [14]:

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load language model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Language model loaded successfully!")
print("Model:", model_name)
print("Device:", next(model.parameters()).device)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Language model loaded successfully!
Model: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda:0


In [16]:

import torch

# Ask a question
question = "What is LangChain?"

# Retrieve relevant document chunks
retrieved_docs = retriever.invoke(question)

# Combine retrieved information
context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

# Prepare the prompt
messages = [
    {
        "role": "system",
        "content": (
            "Answer using only the provided document context. "
            "If the answer is not available, say you don't know."
        )
    },
    {
        "role": "user",
        "content": f"Context:\n{context}\n\nQuestion: {question}"
    }
]

# Convert prompt into model inputs
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(model.device)

# Generate the answer
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode only the newly generated tokens
generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("Question:", question)
print("\nAI Generated Answer:\n", answer)


Question: What is LangChain?

AI Generated Answer:
 LangChain is an open-source framework for developing applications powered by Large Language Models (LLMs).


In [25]:


def ask_rag(question):
    # Retrieve documents with distance scores
    results = vectorstore.similarity_search_with_score(
        question,
        k=3
    )

    # Reject unrelated questions
    if not results or results[0][1] > 1.4:
        return "I don't know based on the provided document."

    # Combine relevant document chunks
    context = "\n\n".join(
        doc.page_content for doc, score in results
        if score <= 1.4
    )

    # Prepare the prompt
    messages = [
        {
            "role": "system",
            "content": (
                "Answer only using the provided document context. "
                "Do not use outside knowledge. "
                "If the answer is not in the context, "
                "say: I don't know based on the provided document."
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }
    ]

    # Prepare model inputs
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    # Generate answer
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode generated answer
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )


# Test the updated function
print("Test 1:", ask_rag("What is LangChain?"))
print("Test 2:", ask_rag("What is the capital of France?"))


Test 1: LangChain is an open-source framework for developing applications powered by Large Language Models (LLMs).
Test 2: I don't know based on the provided document.


In [18]:

%pip install -q gradio


In [27]:

import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma

def process_pdfs(pdf_files):
    global vectorstore

    if not pdf_files:
        return "Please upload at least one PDF."

    all_documents = []

    for pdf_file in pdf_files:
        loader = PyPDFLoader(pdf_file)
        all_documents.extend(loader.load())

    new_chunks = text_splitter.split_documents(all_documents)

    # Create a new database for the uploaded PDFs
    vectorstore = Chroma.from_documents(
        documents=new_chunks,
        embedding=embedding_model,
        collection_name="uploaded_pdfs"
    )

    return (
        f"Successfully processed {len(pdf_files)} PDF(s), "
        f"containing {len(new_chunks)} text chunks."
    )


In [28]:

import gradio as gr

with gr.Blocks(title="Multi-PDF RAG Chatbot") as demo:
    gr.Markdown("# Multi-PDF Question Answering Chatbot")
    gr.Markdown(
        "Upload PDF documents and ask questions using "
        "LangChain, ChromaDB, and Qwen2.5."
    )

    pdf_upload = gr.File(
        label="Upload PDF Documents",
        file_count="multiple",
        file_types=[".pdf"],
        type="filepath"
    )

    process_button = gr.Button("Process PDFs")
    status = gr.Textbox(label="Processing Status")

    gr.Markdown("### Ask Questions")

    question = gr.Textbox(
        label="Your Question",
        placeholder="Ask something about your PDFs..."
    )

    answer = gr.Textbox(label="AI Generated Answer")
    ask_button = gr.Button("Ask Question")

    process_button.click(
        fn=process_pdfs,
        inputs=pdf_upload,
        outputs=status
    )

    ask_button.click(
        fn=ask_rag,
        inputs=question,
        outputs=answer
    )

demo.launch(share=True)



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a2484c1d200e297a66.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [20]:

# Check how closely a question matches the PDF
question = "What is the capital of France?"

results = vectorstore.similarity_search_with_relevance_scores(
    question,
    k=3
)

for doc, score in results:
    print("Relevance score:", round(score, 3))
    print("Text:", doc.page_content[:150])
    print("---")


Relevance score: -0.267
Text: LangChain is an open-source framework for developing applications powered by Large 
Language Models (LLMs). It helps developers connect LLMs with exte
---
Relevance score: -0.355
Text: PUSL 3205-Large Language Models and Generative AI 
 
RAG architecture 
Setting up your python environment 
1. Create a folder in your machine 
2. Navi
---
Relevance score: -0.367
Text: PUSL 3205-Large Language Models and Generative AI 
 
 
 
Build a small chat window and test your model
---


/tmp/ipykernel_379/2468942789.py:4: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='53ced1fd-1271-4d99-9901-f4c0b1b2697a', metadata={'producer': 'Microsoft® Word for Microsoft 365', 'page': 0, 'total_pages': 6, 'page_label': '1', 'author': 'Thilini Bakmeedeniya', 'creationdate': '2026-10-31T09:13:30+05:30', 'creator': 'Microsoft® Word for Microsoft 365', 'source': 'Lab sheet 5.pdf', 'moddate': '2026-10-31T09:13:30+05:30'}, page_content='LangChain is an open-source framework for developing applications powered by Large \nLanguage Models (LLMs). It helps developers connect LLMs with external data \nsources, embedding models, vector databases, retrievers, APIs and other tools. \n \n7. Check the chromaDB \npython -c "import chromadb; print (\'ChromaDB installed successfully\')" \n \n8. Save your dependencies \npip freeze > requirements.txt'), -0.26677849148537836), (Document(id='4a3aaafa-0a9e-462d-9a49-870328268d28', metadata={'producer': 'Microsoft® Word for Micr

In [22]:

questions = [
    "What is LangChain?",
    "What is ChromaDB?",
    "What is the capital of France?"
]

for question in questions:
    results = vectorstore.similarity_search_with_score(
        question,
        k=1
    )

    print("Question:", question)
    print("Distance:", round(results[0][1], 3))
    print("---")


Question: What is LangChain?
Distance: 0.95
---
Question: What is ChromaDB?
Distance: 1.16
---
Question: What is the capital of France?
Distance: 1.791
---
